# 面试问题：Computer-Use Agent 的截图理解、坐标 grounding、动作循环与安全确认怎样设计？

**一句话回答**：每轮把带 revision、viewport、DPR 和窗口信息的 observation 交给模型，模型输出受 schema 约束的 click/type/scroll 等动作；Host 在执行前把归一化坐标映射到当前 frame，检查 frame 未过期、目标可交互、风险/权限和审批，再执行并观察状态 diff。终止依据可验证环境状态，而不是模型自称完成。

本 Notebook 手写 Frame/Box 坐标转换、IoU grounding、stale observation 门禁、风险审批、有界循环、幂等文本输入、taint/secret 防护和执行式评测，不操作真实桌面。


In [ ]:
from dataclasses import dataclass
import hashlib,math
import numpy as np

# 使用虚拟屏幕，保证 Notebook 不产生真实外部副作用。
SEED148=14801
assert SEED148==14801
assert math.isclose(.5*1920,960)
assert len(hashlib.sha256(b"frame").hexdigest())==64


## 1. Observation 必须绑定 frame revision 与坐标空间

截图像素、CSS viewport、设备像素比、窗口偏移和缩放可能不同。动作应携带基于哪个 frame 生成；窗口 resize、页面导航或滚动后旧坐标失效。日志保存敏感区域遮罩后的 frame digest，不默认保存明文屏幕。


In [ ]:
@dataclass(frozen=True)
class Frame148:
    revision:int; width:int; height:int; dpr:float; offset_x:int=0; offset_y:int=0
    def __post_init__(self):
        # 非法尺寸会让所有坐标门禁失去意义，因此立即拒绝。
        if self.width<=0 or self.height<=0 or self.dpr<=0: raise ValueError("frame_contract")
frame148=Frame148(7,1280,720,2.0,100,50)
assert frame148.revision==7
assert frame148.width/frame148.height>1
try: Frame148(1,0,10,1); raise AssertionError("bad frame")
except ValueError as e: assert str(e)=="frame_contract"


## 2. 模型输出归一化坐标，Host 映射并裁剪

归一化 `[0,1]²` 与分辨率无关；Host 映射到当前 viewport，再加窗口偏移。越界值拒绝而不是静默点击屏幕边缘。若自动化后端使用 CSS pixels，还需明确 DPR 是否参与换算，避免双乘。


In [ ]:
def map_point148(nx,ny,frame):
    # 这里约定 width/height 已是执行器坐标，不再重复乘 DPR。
    if not 0<=nx<=1 or not 0<=ny<=1: raise ValueError("normalized_coordinate")
    return frame.offset_x+round(nx*(frame.width-1)),frame.offset_y+round(ny*(frame.height-1))
point148=map_point148(.5,.25,frame148)
assert point148==(740,230)
assert map_point148(0,0,frame148)==(100,50)
try: map_point148(1.2,.5,frame148); raise AssertionError("out of bounds")
except ValueError as e: assert str(e)=="normalized_coordinate"


## 3. Grounding 用 box/role/text 与可交互性共同验证

视觉模型给候选框，accessibility tree 可给 role/name；融合时用 IoU/中心距离和语义匹配。点击前检查点位落在目标框、元素未 disabled/遮挡。只凭 OCR 文本可能点到页面中的恶意复制提示。


In [ ]:
@dataclass(frozen=True)
class Box148:
    x1:float; y1:float; x2:float; y2:float
    @property
    def center(self): return ((self.x1+self.x2)/2,(self.y1+self.y2)/2)
def iou148(a,b):
    # 交集面积除以并集面积，用于视觉框和结构树框对齐。
    iw=max(0,min(a.x2,b.x2)-max(a.x1,b.x1)); ih=max(0,min(a.y2,b.y2)-max(a.y1,b.y1)); inter=iw*ih
    area=lambda z:max(0,z.x2-z.x1)*max(0,z.y2-z.y1)
    return inter/(area(a)+area(b)-inter or 1)
visual148=Box148(100,100,200,150); ax148=Box148(105,98,198,152)
assert iou148(visual148,ax148)>.8
assert visual148.center==(150,125)
assert iou148(visual148,Box148(500,500,600,600))==0


## 4. Action 执行采用 observation revision 前置条件

模型思考期间页面可能自动更新；执行器比较 action.frame_revision 与当前 frame，若不等则重新截图和规划。对目标元素还可绑定 accessibility node ID/文本 hash。不能把“通常页面不变”当一致性保证。


In [ ]:
@dataclass(frozen=True)
class Action148:
    kind:str; frame_revision:int; payload:dict
def precondition148(action,current_revision):
    # 旧截图生成的动作必须中止，不做猜测性坐标修正。
    if action.frame_revision!=current_revision: raise RuntimeError("stale_observation")
    return True
click148=Action148("click",7,{"point":point148})
assert precondition148(click148,7)
try: precondition148(click148,8); raise AssertionError("stale click")
except RuntimeError as e: assert str(e)=="stale_observation"
assert click148.kind=="click"


## 5. 风险分类在执行器，不交给页面或模型自报

浏览/滚动通常低风险；提交付款、发送消息、删除、公开发布、下载执行文件属于高风险，需 capability、预览和 Human-in-the-Loop。审批票据绑定规范化动作、目标、金额/内容、frame revision 与过期时间，参数变化后重新审批。


In [ ]:
RISK148={"scroll":"low","focus":"low","type":"medium","submit-payment":"high","delete":"high","send":"high"}
def may_execute148(action,capabilities,approved_digest=None):
    # 高风险动作同时要求专用 capability 和与动作内容绑定的审批 digest。
    risk=RISK148.get(action.kind,"high"); digest=hashlib.sha256(repr(action).encode()).hexdigest()
    return risk!="high" or (action.kind in capabilities and approved_digest==digest)
pay148=Action148("submit-payment",7,{"amount":100,"currency":"CNY"}); pay_digest148=hashlib.sha256(repr(pay148).encode()).hexdigest()
assert not may_execute148(pay148,{"submit-payment"})
assert may_execute148(pay148,{"submit-payment"},pay_digest148)
assert may_execute148(Action148("scroll",7,{"dy":300}),set())


## 6. Observe—Act Loop 有预算、循环检测和环境终止条件

每步记录 frame digest、动作、结果、页面状态 diff 与预算。相同 `(frame_digest, action)` 重复出现说明卡住，应重新定位或停止。最大步数、deadline、连续无变化和高风险等待都是确定性终止条件。


In [ ]:
def loop_guard148(trace,max_steps):
    # 相同观察上重复同一动作两次即报告 cycle。
    if len(trace)>=max_steps: return "budget"
    keys=[(x["frame"],x["action"]) for x in trace]
    return "cycle" if len(keys)!=len(set(keys)) else "continue"
trace148=[{"frame":"f1","action":"click:save"},{"frame":"f2","action":"scroll"}]
assert loop_guard148(trace148,5)=="continue"
assert loop_guard148(trace148+[trace148[0]],5)=="cycle"
assert loop_guard148(trace148,2)=="budget"


## 7. 输入动作尽量幂等，并验证 postcondition

`type("abc")` 重试可能变成 `abcabc`；更安全的是 `set_value(expected_old, desired)`，旧值不符就重新观察。点击提交使用业务 idempotency key，并在执行后检查订单/文件/消息状态，而不是只看到按钮消失就宣布成功。


In [ ]:
def set_value148(state,field,expected,desired):
    # compare-and-set 防止超时重试重复追加文本或覆盖并发修改。
    if state.get(field)!=expected: return False
    state[field]=desired; return True
form148={"email":""}
assert set_value148(form148,"email","","a@example.com")
assert not set_value148(form148,"email","","a@example.com")
assert form148["email"]=="a@example.com"


## 8. 执行式评测验证最终环境状态与安全轨迹

相同任务可能有多条合法点击路径，不能只做 action exact match。重置可复现环境后检查数据库/文件/UI 最终状态、禁止副作用、步骤/时间与审批；把视觉 grounding、动作选择、执行错误和环境漂移分开归因。


In [ ]:
before148={"draft":True,"sent":False,"recipients":[]}; after148={"draft":False,"sent":True,"recipients":["alice"]}
def task_oracle148(state):
    # 最终状态 oracle 不要求唯一动作序列，但明确禁止额外收件人。
    return state["sent"] and not state["draft"] and state["recipients"]==["alice"]
assert not task_oracle148(before148)
assert task_oracle148(after148)
assert not task_oracle148({**after148,"recipients":["alice","mallory"]})


## 面试总结

完整链路是：**截图带 revision/viewport/DPR → 归一化坐标映射 → 视觉框与 accessibility role 融合 → 执行前检查 stale frame/可交互性 → Host 风险分类与绑定参数的审批 → 有界 observe-act、无变化/cycle stop → compare-and-set/idempotency/postcondition → 页面内容 taint 与 secret 遮罩 → 重置环境后的最终状态和禁止副作用 oracle**。

延伸阅读：[OSWorld](https://arxiv.org/abs/2404.07972)、[Mind2Web](https://arxiv.org/abs/2306.06070)、[WebArena](https://arxiv.org/abs/2307.13854)。
